In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cartopy.crs as ccrs
import cmocean
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr 
from scipy.stats import shapiro 
from scipy.stats import linregress 
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from OceanDataStore import OceanDataCatalog 
from nemo_cookbook import NEMODataTree  
from matplotlib.patches import Rectangle


In [2]:
def select_by_latlon(ds, lat_target, lon_target):
    lat = ds['gphit'].values
    lon = ds['glamt'].values
    distance = np.sqrt((lat - lat_target)**2 + (lon - lon_target)**2)
    y_idx, x_idx = np.unravel_index(np.argmin(distance), distance.shape)
    print('x index =', x_idx)
    print('y index =', y_idx)
    return ds.isel(y=y_idx, x=x_idx)

In [3]:
# Open model and config files

catalog = OceanDataCatalog(catalog_name='noc-model-stac')
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')
ds1_annual = catalog.open_dataset(id=catalog.Items[0].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12')

catalog.search(collection='noc-npd-era5', item_name='domain_cfg')
config = catalog.open_dataset(id=catalog.Items[1].id)

# Merge into a data tree
datasets_annual = {'parent': {'domain': config, 'gridT': ds1_annual}}
dt_global_annual = NEMODataTree.from_datasets(datasets = datasets_annual)

# Clip to North Atlantic 
bbox = (-85.0, 0.0, 0.0, 80.0)
dt_annual = dt_global_annual.clip_grid(grid='/gridT', bbox=bbox)

# Convert to datasets
ds_annual = (dt_annual['/gridT']).dataset

MaxRetryError: HTTPSConnectionPool(host='noc-msm-o.s3-ext.jc.rl.ac.uk', port=443): Max retries exceeded with url: /oceandatastore/noc-model-stac/catalog.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='noc-msm-o.s3-ext.jc.rl.ac.uk', port=443) at 0x28b5314efd0>, 'Connection to noc-msm-o.s3-ext.jc.rl.ac.uk timed out. (connect timeout=None)'))

In [4]:
salinity_0 = (ds_annual['so_abs'].sel(deptht = 0, method = 'nearest')).compute()
salinity_0['time_counter'] = salinity_0['time_counter'].dt.year

ny, nx = salinity_0.sizes['j'], salinity_0.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_0.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_0['j'],
        'i': salinity_0['i'],
        'gphit': (('j', 'i'), salinity_0['gphit'].values),
        'glamt': (('j', 'i'), salinity_0['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of sea-surface salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_0['j'],
        'i': salinity_0['i'],
        'gphit': (('j', 'i'), salinity_0['gphit'].values),
        'glamt': (('j', 'i'), salinity_0['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of sea-surface salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('Sea-surface Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
## 50m

salinity_50 = (ds_annual['so_abs'].sel(deptht = 50, method = 'nearest')).compute()
salinity_50['time_counter'] = salinity_50['time_counter'].dt.year

ny, nx = salinity_50.sizes['j'], salinity_50.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_50.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_50['j'],
        'i': salinity_50['i'],
        'gphit': (('j', 'i'), salinity_50['gphit'].values),
        'glamt': (('j', 'i'), salinity_50['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 50m salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_50['j'],
        'i': salinity_50['i'],
        'gphit': (('j', 'i'), salinity_50['gphit'].values),
        'glamt': (('j', 'i'), salinity_50['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 50m salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('50m Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
## 100m

salinity_100 = (ds_annual['so_abs'].sel(deptht = 100, method = 'nearest')).compute()
salinity_100['time_counter'] = salinity_100['time_counter'].dt.year

ny, nx = salinity_100.sizes['j'], salinity_100.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_100.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_100['j'],
        'i': salinity_100['i'],
        'gphit': (('j', 'i'), salinity_100['gphit'].values),
        'glamt': (('j', 'i'), salinity_100['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 100m salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_100['j'],
        'i': salinity_100['i'],
        'gphit': (('j', 'i'), salinity_100['gphit'].values),
        'glamt': (('j', 'i'), salinity_100['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 100m salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('100m Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
## 150m 

salinity_150 = (ds_annual['so_abs'].sel(deptht = 150, method = 'nearest')).compute()
salinity_150['time_counter'] = salinity_150['time_counter'].dt.year

ny, nx = salinity_150.sizes['j'], salinity_150.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_150.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_150['j'],
        'i': salinity_150['i'],
        'gphit': (('j', 'i'), salinity_150['gphit'].values),
        'glamt': (('j', 'i'), salinity_150['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 150m salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_150['j'],
        'i': salinity_150['i'],
        'gphit': (('j', 'i'), salinity_150['gphit'].values),
        'glamt': (('j', 'i'), salinity_150['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 150m salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('150m Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
salinity_200 = (ds_annual['so_abs'].sel(deptht = 200, method = 'nearest')).compute()
salinity_200['time_counter'] = salinity_200['time_counter'].dt.year

ny, nx = salinity_200.sizes['j'], salinity_200.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_200.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_200['j'],
        'i': salinity_200['i'],
        'gphit': (('j', 'i'), salinity_200['gphit'].values),
        'glamt': (('j', 'i'), salinity_200['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 200m salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_200['j'],
        'i': salinity_200['i'],
        'gphit': (('j', 'i'), salinity_200['gphit'].values),
        'glamt': (('j', 'i'), salinity_200['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 200m salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('200m Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
## 500m 

salinity_500 = (ds_annual['so_abs'].sel(deptht = 500, method = 'nearest')).compute()
salinity_500['time_counter'] = salinity_500['time_counter'].dt.year

ny, nx = salinity_500.sizes['j'], salinity_500.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_500.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_500['j'],
        'i': salinity_500['i'],
        'gphit': (('j', 'i'), salinity_500['gphit'].values),
        'glamt': (('j', 'i'), salinity_500['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 500m salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_500['j'],
        'i': salinity_500['i'],
        'gphit': (('j', 'i'), salinity_500['gphit'].values),
        'glamt': (('j', 'i'), salinity_500['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 500m salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('500m Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
## 1km 

salinity_1000 = (ds_annual['so_abs'].sel(deptht = 1000, method = 'nearest')).compute()
salinity_1000['time_counter'] = salinity_1000['time_counter'].dt.year

ny, nx = salinity_1000.sizes['j'], salinity_1000.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_1000.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_1000['j'],
        'i': salinity_1000['i'],
        'gphit': (('j', 'i'), salinity_1000['gphit'].values),
        'glamt': (('j', 'i'), salinity_1000['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 1km salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_1000['j'],
        'i': salinity_1000['i'],
        'gphit': (('j', 'i'), salinity_1000['gphit'].values),
        'glamt': (('j', 'i'), salinity_1000['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of 1km salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('1km Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)


In [ ]:
## 2km

salinity_2000 = (ds_annual['so_abs'].sel(deptht = 2000, method = 'nearest')).compute()
salinity_2000['time_counter'] = salinity_2000['time_counter'].dt.year

ny, nx = salinity_2000.sizes['j'], salinity_2000.sizes['i']
trend_data = np.full((ny, nx), np.nan, dtype = np.float32)
pval_data = np.full((ny, nx), np.nan, dtype = np.float32) 

for j_idx in range (ny):
    for i_idx in range (nx):
        point = salinity_2000.isel(j=j_idx, i=i_idx)
        if np.isnan(point).all():
            continue
        Y = point.values
        X = sm.add_constant(point['time_counter'].values)
        model = sm.GLSAR(Y, X, 1)
        results = model.iterative_fit(maxiter=50)
        trend_data[j_idx, i_idx] = results.params[1] 
        pval_data[j_idx, i_idx] = results.pvalues[1]

trend_da = xr.DataArray(data = trend_data, dims=['j', 'i'], 
        coords={ 'j': salinity_2000['j'],
        'i': salinity_2000['i'],
        'gphit': (('j', 'i'), salinity_2000['gphit'].values),
        'glamt': (('j', 'i'), salinity_2000['glamt'].values)}, name='trend',
        attrs={'description': 'Linear trend over time of 2km salinity'})

pval_da = xr.DataArray(data = pval_data, dims=['j', 'i'], 
        coords={ 'j': salinity_2000['j'],
        'i': salinity_2000['i'],
        'gphit': (('j', 'i'), salinity_2000['gphit'].values),
        'glamt': (('j', 'i'), salinity_0['glamt'].values)}, name='trend',
        attrs={'description': 'P-value for linear trend over time of  2km salinity'})

sig_mask = pval_da < 0.05
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.pcolormesh(trend_da['glamt'], trend_da['gphit'], trend_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -0.03, vmax = 0.03)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}
significance = ax.contourf(sig_mask['glamt'], sig_mask['gphit'], sig_mask, levels=[0.5, 1.0], colors='none', hatches=['xxx'], transform=ccrs.PlateCarree())
title = ax.set_title('2km Salinity Trends')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, shrink = 0.3)
